In [1]:
import json
import pandas as pd
import sqlite3

In [2]:
## Q1. The console exporter prints every finished span as a dictionary. 
## Count the spans in the console output - each one is a separate ReadableSpan entry.
## How many spans does the trace produce?

In [3]:
op = !uv run python starter.py "How does the agentic loop keep calling the model until it stops?"
op

['{',
 '    "name": "search",',
 '    "context": {',
 '        "trace_id": "0x341bd8826486fea92228924896a675a4",',
 '        "span_id": "0x82420dfc2b5ae5ba",',
 '        "trace_state": "[]"',
 '    },',
 '    "kind": "SpanKind.INTERNAL",',
 '    "parent_id": "0x0fac0d43f4f5f44d",',
 '    "start_time": "2026-07-20T03:51:44.358816Z",',
 '    "end_time": "2026-07-20T03:51:44.360026Z",',
 '    "status": {',
 '        "status_code": "UNSET"',
 '    },',
 '    "attributes": {',
 '        "search_time": 0.00119781494140625',
 '    },',
 '    "events": [],',
 '    "links": [],',
 '    "resource": {',
 '        "attributes": {',
 '            "telemetry.sdk.language": "python",',
 '            "telemetry.sdk.name": "opentelemetry",',
 '            "telemetry.sdk.version": "1.44.0",',
 '            "service.instance.id": "42cb66f0-f8d0-42dd-91c3-7ef4bad2a876",',
 '            "service.name": "unknown_service"',
 '        },',
 '        "schema_url": ""',
 '    }',
 '}',
 '{',
 '    "name": "llm"

In [4]:
### 3

In [5]:
## Q2. Now re-run the query. How many input tokens do we see?

In [6]:
"""
"attributes": {',
 '        "model": "gpt-5.4-mini",',
 '        "instruction": "\\nYour task is to answer questions from the course participants\\nbased on the provided context.\\n\\nUse the context to find relevant information and provide accurate\\nanswers. If the answer is not found in the context,\\nrespond with \\"I don\'t know.\\"\\n",',
 '        "answer": "It keeps calling the model inside a `while True` loop.\\n\\nEach iteration:\\n1. send the full `messages` history to the model,\\n2. check the response for any `function_call` items,\\n3. run those tools and append the tool outputs to `messages`,\\n4. if there were no function calls, `break` out of the loop.\\n\\nSo the stop condition is simple: **when the model returns a response with no function calls, the loop ends.**",',
 '        "response_time": 2.306572914123535,',
 '        "input_tokens": 7111,',
 '        "output_tokens": 101,',
 '        "total_tokens": 7212,',
 '        "cost": 0.00578775',
 '    }
"""

'\n"attributes": {\',\n \'        "model": "gpt-5.4-mini",\',\n \'        "instruction": "\\nYour task is to answer questions from the course participants\\nbased on the provided context.\\n\\nUse the context to find relevant information and provide accurate\\nanswers. If the answer is not found in the context,\\nrespond with \\"I don\'t know.\\"\\n",\',\n \'        "answer": "It keeps calling the model inside a `while True` loop.\\n\\nEach iteration:\\n1. send the full `messages` history to the model,\\n2. check the response for any `function_call` items,\\n3. run those tools and append the tool outputs to `messages`,\\n4. if there were no function calls, `break` out of the loop.\\n\\nSo the stop condition is simple: **when the model returns a response with no function calls, the loop ends.**",\',\n \'        "response_time": 2.306572914123535,\',\n \'        "input_tokens": 7111,\',\n \'        "output_tokens": 101,\',\n \'        "total_tokens": 7212,\',\n \'        "cost": 0.005787

In [7]:
### 7111
### 7000

In [8]:
## Q3. For a typical query, roughly how long does the LLM call take?

In [9]:
2.306572914123535*1000

2306.572914123535

In [10]:
### Over 2000ms

In [11]:
## Q4. Re-run the query from Q1. Which span names appear in the spans table?

In [23]:
conn = sqlite3.connect("traces.db")
query = "SELECT * FROM spans"
df = pd.read_sql_query(query, conn)
conn.close()

In [24]:
df

,name,start_time,end_time,input_tokens,output_tokens,cost
0,search,1784519644080306000,1784519644081854000,NaN,NaN,NaN
1,llm,1784519644082639000,1784519646801197000,7111.0,87.0,0.005725
2,rag,1784519644080253000,1784519646801935000,NaN,NaN,NaN
3,search,1784519880352980000,1784519880354209000,NaN,NaN,NaN
4,llm,1784519880354751000,1784519882537304000,7111.0,101.0,0.005788
5,rag,1784519880352939000,1784519882538066000,NaN,NaN,NaN
6,search,1784519889741185000,1784519889742013000,NaN,NaN,NaN
7,llm,1784519889742596000,1784519891632076000,7111.0,108.0,0.005819
8,rag,1784519889741146000,1784519891633150000,NaN,NaN,NaN
9,search,1784519897509428000,1784519897510375000,NaN,NaN,NaN


In [25]:
len(df)

12

In [26]:
### rag, search, and llm

In [27]:
## Q5. Using SQL (or pandas), compute the total duration for each span name excluding rag. 
## Which span type takes the most total time?

In [28]:
df['duration'] = df['end_time'] - df['start_time']
df

,name,start_time,end_time,input_tokens,output_tokens,cost,duration
0,search,1784519644080306000,1784519644081854000,NaN,NaN,NaN,1548000
1,llm,1784519644082639000,1784519646801197000,7111.0,87.0,0.005725,2718558000
2,rag,1784519644080253000,1784519646801935000,NaN,NaN,NaN,2721682000
3,search,1784519880352980000,1784519880354209000,NaN,NaN,NaN,1229000
4,llm,1784519880354751000,1784519882537304000,7111.0,101.0,0.005788,2182553000
5,rag,1784519880352939000,1784519882538066000,NaN,NaN,NaN,2185127000
6,search,1784519889741185000,1784519889742013000,NaN,NaN,NaN,828000
7,llm,1784519889742596000,1784519891632076000,7111.0,108.0,0.005819,1889480000
8,rag,1784519889741146000,1784519891633150000,NaN,NaN,NaN,1892004000
9,search,1784519897509428000,1784519897510375000,NaN,NaN,NaN,947000


In [30]:
average_duration = df.groupby('name')['duration'].mean()
average_duration

name
llm       2.161421e+09
rag       2.164242e+09
search    1.138000e+06
Name: duration, dtype: float64

In [32]:
average_duration['llm'] > average_duration['search']

np.True_

In [33]:
average_duration['llm'] - average_duration['search']

np.float64(2160283250.0)

In [34]:
### llm

In [37]:
## Q6. How much do the input tokens vary across these 4 runs?

In [38]:
df['input_tokens']

0        NaN
1     7111.0
2        NaN
3        NaN
4     7111.0
5        NaN
6        NaN
7     7111.0
8        NaN
9        NaN
10    7111.0
11       NaN
Name: input_tokens, dtype: float64

In [40]:
### They're identical